# Étteremértékelések elemzése – 03B. notebook
## Application-Compatible Aspect-Level Sentiment Classification

**Adat:** SemEval-2016 Task 5 – English Restaurants, Subtask 1  
**Modell:** DistilBERT

## Miért van szükség erre a verzióra?

A korábbi sentiment classifier bemenete:

**sentence + target + aspect → sentiment**

A valós Google Local review-kban azonban nincs kézzel annotált `target` mező. Az Aspect Detection modellünk csak az aspektuskategóriát adja vissza.

Ezért a végső alkalmazási pipeline-hoz olyan sentiment modellt készítünk, amely közvetlenül használható:

**sentence + aspect → positive / neutral / negative**

Ez továbbra is aspektusszintű sentiment classification.

### Adattisztítás

Ha ugyanabban a mondatban ugyanahhoz az aspektushoz több, egymásnak ellentmondó polarity tartozik, akkor a `sentence + aspect` bemenet önmagában nem képes egyértelműen megkülönböztetni őket.

Ezért az ilyen ritka, ambivalens eseteket kizárjuk a tanításból.

A SemEval training adaton:

- kb. **2216** tiszta `sentence + aspect` példa marad;
- kb. **42** ambivalens `sentence + aspect` kulcsot zárunk ki.

A modellezési logika ugyanaz marad, mint korábban:

1. review-szintű train/validation split;
2. baseline DistilBERT;
3. class-weighted DistilBERT;
4. macro-F1 alapú modellválasztás;
5. végleges modell mentése és ZIP export.


## 1. Csomagok és GPU

In [ ]:
!pip -q install "transformers>=4.46,<5" datasets accelerate scikit-learn


In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

pd.set_option("display.max_colwidth", 180)
RANDOM_STATE = 42

print("PyTorch:", torch.__version__)
print("GPU elérhető:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Training XML feltöltése

Töltsd fel:

`ABSA16_Restaurants_Train_SB1_v2.xml`


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Nem Colab környezet.")


In [ ]:
TRAIN_PATH = Path("ABSA16_Restaurants_Train_SB1_v2.xml")
assert TRAIN_PATH.exists(), f"Hiányzó fájl: {TRAIN_PATH}"
print("Training XML rendben.")


## 3. Opinion annotációk beolvasása

In [ ]:
def parse_opinions(path):
    root = ET.parse(path).getroot()
    rows = []

    for review in root.findall("Review"):
        review_id = review.attrib["rid"]
        sentences = review.find("sentences")

        if sentences is None:
            continue

        for sentence in sentences.findall("sentence"):
            sentence_id = sentence.attrib["id"]
            text = sentence.findtext("text", default="")
            out_of_scope = (
                sentence.attrib.get("OutOfScope", "FALSE").upper() == "TRUE"
            )

            opinions = sentence.find("Opinions")
            if opinions is None:
                continue

            for opinion in opinions.findall("Opinion"):
                rows.append({
                    "review_id": review_id,
                    "sentence_id": sentence_id,
                    "text": text,
                    "category": opinion.attrib.get("category"),
                    "polarity": opinion.attrib.get("polarity"),
                    "out_of_scope": out_of_scope,
                })

    return pd.DataFrame(rows)

opinions_df = parse_opinions(TRAIN_PATH)

print("Opinion annotációk:", len(opinions_df))
display(opinions_df.head(10))


## 4. `sentence + aspect` sentiment tábla létrehozása

Egy adott mondat és aspektus több targethez is tartozhat.

Ha ezek sentimentje azonos, egyetlen tanítópéldává vonjuk össze őket.

Ha ugyanahhoz a `sentence + aspect` kulcshoz **különböző polarityk** tartoznak, azt ambivalens esetnek tekintjük és kizárjuk.


In [ ]:
def build_application_sentiment_df(opinions_df):
    df = opinions_df.loc[
        ~opinions_df["out_of_scope"]
    ].copy()

    # Azonos sentence + aspect + polarity ismétlések összevonása
    df = df.drop_duplicates(
        subset=[
            "review_id",
            "sentence_id",
            "text",
            "category",
            "polarity",
        ]
    ).copy()

    key_cols = [
        "review_id",
        "sentence_id",
        "category",
    ]

    polarity_counts = (
        df.groupby(key_cols)["polarity"]
        .nunique()
        .rename("n_polarities")
        .reset_index()
    )

    ambiguous_keys = polarity_counts.loc[
        polarity_counts["n_polarities"] > 1,
        key_cols
    ].copy()

    if len(ambiguous_keys) > 0:
        marker = ambiguous_keys.assign(ambiguous=True)

        df = df.merge(
            marker,
            on=key_cols,
            how="left",
        )

        df = df.loc[
            df["ambiguous"].isna()
        ].drop(columns="ambiguous")

    df = (
        df.drop_duplicates(
            subset=key_cols
        )
        .reset_index(drop=True)
    )

    return df, ambiguous_keys


sentiment_df, ambiguous_keys = build_application_sentiment_df(
    opinions_df
)

print("Tiszta sentence + aspect példák:", len(sentiment_df))
print("Kizárt ambivalens kulcsok:", len(ambiguous_keys))

display(sentiment_df.head(10))


## 5. Sentiment eloszlás

In [ ]:
distribution = (
    sentiment_df["polarity"]
    .value_counts()
    .rename_axis("polarity")
    .reset_index(name="count")
)

distribution["ratio"] = (
    distribution["count"]
    / len(sentiment_df)
)

display(distribution)


## 6. Modell input

A végső input:

`Sentence: ... Aspect: ...`

A target mezőt teljesen elhagyjuk, így ugyanezt a formátumot tudjuk majd használni a Google review-knál.


In [ ]:
def build_model_input(row):
    return (
        f"Sentence: {row['text']} "
        f"Aspect: {row['category']}."
    )

sentiment_df["model_input"] = sentiment_df.apply(
    build_model_input,
    axis=1,
)

display(
    sentiment_df[
        ["model_input", "polarity"]
    ].head(10)
)


## 7. Címkék

In [ ]:
LABELS = [
    "negative",
    "neutral",
    "positive",
]

label2id = {
    label: i
    for i, label in enumerate(LABELS)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

sentiment_df["label"] = (
    sentiment_df["polarity"]
    .map(label2id)
)

print(label2id)


## 8. Review-szintű train / validation split

Ugyanazon review opinionjei nem kerülhetnek egyszerre train és validation mintába.


In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, val_idx = next(
    splitter.split(
        sentiment_df,
        groups=sentiment_df["review_id"],
    )
)

train_df = (
    sentiment_df.iloc[train_idx]
    .reset_index(drop=True)
)

val_df = (
    sentiment_df.iloc[val_idx]
    .reset_index(drop=True)
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Train review-k:", train_df["review_id"].nunique())
print("Validation review-k:", val_df["review_id"].nunique())

overlap = (
    set(train_df["review_id"])
    & set(val_df["review_id"])
)

print("Review overlap:", len(overlap))
assert len(overlap) == 0


In [ ]:
display(
    pd.DataFrame({
        "train_count": train_df["polarity"].value_counts(),
        "train_ratio": train_df["polarity"].value_counts(normalize=True),
        "val_count": val_df["polarity"].value_counts(),
        "val_ratio": val_df["polarity"].value_counts(normalize=True),
    }).fillna(0)
)


## 9. Hugging Face Dataset

In [ ]:
train_hf = (
    train_df[
        ["model_input", "label"]
    ]
    .rename(
        columns={
            "model_input": "text",
            "label": "labels",
        }
    )
)

val_hf = (
    val_df[
        ["model_input", "label"]
    ]
    .rename(
        columns={
            "model_input": "text",
            "label": "labels",
        }
    )
)

train_dataset = Dataset.from_pandas(
    train_hf,
    preserve_index=False,
)

val_dataset = Dataset.from_pandas(
    val_hf,
    preserve_index=False,
)

print(train_dataset)
print(val_dataset)


## 10. DistilBERT és tokenizálás

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=160,
    )

train_tokenized = train_dataset.map(
    tokenize_batch,
    batched=True,
)

val_tokenized = val_dataset.map(
    tokenize_batch,
    batched=True,
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


## 11. Értékelési metrikák

In [ ]:
def compute_sentiment_metrics(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        preds
    )

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0,
    )

    _, _, f1_weighted, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0,
    )

    return {
        "accuracy": accuracy,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
    }


# 12. Baseline modell

In [ ]:
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

baseline_args = TrainingArguments(
    output_dir="sentiment_application_baseline_output",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=RANDOM_STATE,
)

baseline_trainer = Trainer(
    model=baseline_model,
    args=baseline_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_sentiment_metrics,
)

baseline_trainer.train()


## 13. Baseline eredmények

In [ ]:
baseline_metrics = baseline_trainer.evaluate()

baseline_output = baseline_trainer.predict(
    val_tokenized
)

val_labels = baseline_output.label_ids.astype(int)
baseline_preds = np.argmax(
    baseline_output.predictions,
    axis=-1
)

p, r, f1, support = precision_recall_fscore_support(
    val_labels,
    baseline_preds,
    labels=list(range(len(LABELS))),
    average=None,
    zero_division=0,
)

baseline_per_class = pd.DataFrame({
    "sentiment": LABELS,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support": support,
})

print("Baseline macro-F1:", baseline_metrics["eval_f1_macro"])
display(baseline_per_class)


# 14. Class-weighted modell

A neutral osztály ritkasága miatt inverse-frequency class weightet használunk.


In [ ]:
train_class_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
    .reindex(
        range(len(LABELS)),
        fill_value=0
    )
    .values
)

n_train = len(train_df)
n_classes = len(LABELS)

class_weights = (
    n_train
    / (n_classes * train_class_counts)
)

class_weight_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
)

display(
    pd.DataFrame({
        "sentiment": LABELS,
        "train_examples": train_class_counts,
        "class_weight": class_weights,
    })
)


In [ ]:
from torch.nn import CrossEntropyLoss

class WeightedSentimentTrainer(Trainer):

    def __init__(
        self,
        *args,
        class_weights=None,
        **kwargs
    ):
        super().__init__(
            *args,
            **kwargs
        )
        self.class_weights = class_weights

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs["labels"]

        model_inputs = {
            key: value
            for key, value in inputs.items()
            if key != "labels"
        }

        outputs = model(
            **model_inputs
        )

        logits = outputs.logits

        loss_fn = CrossEntropyLoss(
            weight=self.class_weights.to(
                logits.device
            )
        )

        loss = loss_fn(
            logits,
            labels.long()
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )


In [ ]:
weighted_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

weighted_args = TrainingArguments(
    output_dir="sentiment_application_weighted_output",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=RANDOM_STATE,
)

weighted_trainer = WeightedSentimentTrainer(
    model=weighted_model,
    args=weighted_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_sentiment_metrics,
    class_weights=class_weight_tensor,
)

weighted_trainer.train()


## 15. Weighted modell eredményei

In [ ]:
weighted_metrics = weighted_trainer.evaluate()

weighted_output = weighted_trainer.predict(
    val_tokenized
)

weighted_preds = np.argmax(
    weighted_output.predictions,
    axis=-1
)

p, r, f1, support = precision_recall_fscore_support(
    val_labels,
    weighted_preds,
    labels=list(range(len(LABELS))),
    average=None,
    zero_division=0,
)

weighted_per_class = pd.DataFrame({
    "sentiment": LABELS,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support": support,
})

print("Weighted macro-F1:", weighted_metrics["eval_f1_macro"])
display(weighted_per_class)


## 16. Modell-összehasonlítás és végleges választás

In [ ]:
comparison = pd.DataFrame([
    {
        "model": "Baseline",
        "accuracy": baseline_metrics["eval_accuracy"],
        "macro_f1": baseline_metrics["eval_f1_macro"],
        "weighted_f1": baseline_metrics["eval_f1_weighted"],
    },
    {
        "model": "Class-weighted",
        "accuracy": weighted_metrics["eval_accuracy"],
        "macro_f1": weighted_metrics["eval_f1_macro"],
        "weighted_f1": weighted_metrics["eval_f1_weighted"],
    },
])

display(
    comparison.sort_values(
        "macro_f1",
        ascending=False
    )
)

if weighted_metrics["eval_f1_macro"] > baseline_metrics["eval_f1_macro"]:
    final_trainer = weighted_trainer
    final_model = weighted_trainer.model
    FINAL_MODEL_TYPE = "class-weighted"
else:
    final_trainer = baseline_trainer
    final_model = baseline_trainer.model
    FINAL_MODEL_TYPE = "baseline"

print("Végleges modell:", FINAL_MODEL_TYPE)


## 17. Végleges predikciós függvény

In [ ]:
def predict_sentiment(
    sentence,
    aspect,
):
    final_model.eval()

    text = (
        f"Sentence: {sentence} "
        f"Aspect: {aspect}."
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=160,
    )

    inputs = {
        key: value.to(final_model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        logits = final_model(
            **inputs
        ).logits[0]

    probabilities = torch.softmax(
        logits,
        dim=-1
    ).cpu().numpy()

    return (
        pd.DataFrame({
            "sentiment": LABELS,
            "probability": probabilities,
        })
        .sort_values(
            "probability",
            ascending=False
        )
    )


In [ ]:
predict_sentiment(
    sentence="The coffee was excellent, but the service was very slow.",
    aspect="DRINKS#QUALITY",
)


In [ ]:
predict_sentiment(
    sentence="The coffee was excellent, but the service was very slow.",
    aspect="SERVICE#GENERAL",
)


# 18. Végleges alkalmazási modell mentése és letöltése

A modellt ZIP-be csomagoljuk, mert ezt fogjuk feltölteni az **05 – Place Profile Generation** notebookba.


In [ ]:
import shutil

FINAL_SENTIMENT_MODEL_PATH = "sentiment_distilbert_application"

final_trainer.save_model(
    FINAL_SENTIMENT_MODEL_PATH
)

tokenizer.save_pretrained(
    FINAL_SENTIMENT_MODEL_PATH
)

ZIP_BASE = "sentiment_distilbert_application"

shutil.make_archive(
    ZIP_BASE,
    "zip",
    FINAL_SENTIMENT_MODEL_PATH
)

print("Elmentve:", FINAL_SENTIMENT_MODEL_PATH)
print("ZIP elkészült:", ZIP_BASE + ".zip")

try:
    from google.colab import files
    files.download(ZIP_BASE + ".zip")
except ImportError:
    print("A ZIP a notebook mappájában található.")


# Következő lépés

Ha megvan:

- `aspect_distilbert_weighted.zip`
- `sentiment_distilbert_application.zip`
- `selected_places.csv`
- `selected_reviews.csv`
- `selected_review_sentences.csv`

akkor jöhet az **05 – Place Profile Generation**:

**7395 review sentence → aspect prediction → sentiment prediction → helyenkénti aggregálás → place profile**
